# Gap filling híbrido para Eddy Covariance, versión final v4

Este notebook implementa una etapa independiente de preparación y relleno de huecos para una serie de tiempo Eddy Covariance. La salida está pensada para alimentar un notebook posterior de análisis, evitando que artefactos de imputación contaminen PSD, FI, grafos o regímenes.

## Qué hace

1. Carga el archivo de prueba `carbon_flux_eddy.csv`.
2. Regulariza la malla temporal.
3. Detecta la estructura general de la serie usando disponibilidad combinada de variables objetivo.
4. Aplica un esquema híbrido de gap filling:
   - interpolación temporal limitada para huecos cortos,
   - Random Forest con variables ancla para huecos medianos,
   - Random Forest temporal de respaldo cuando no hay suficientes anclas,
   - no imputación cuando el hueco sigue siendo demasiado problemático.
5. Genera figuras antes y después del llenado.
6. Produce métricas de validación por enmascaramiento.
7. Exporta una tabla global rellenada, flags de calidad y una carpeta con segmentos listos para el notebook de análisis.

## Salidas principales

- `art_regularized.csv`
- `art_segment_summary.csv`
- `art_gapfilled_all_segments.csv`
- `art_gapfill_flags.csv`
- `art_gapfill_validation_summary.csv`
- `analysis_ready_segments/`
- figuras por variable antes y después del llenado
- figuras de métricas de validación


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


## 1. Configuración

In [ ]:
# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

INPUT_CSV = Path("/mnt/data/carbon_flux_eddy.csv")
OUTPUT_DIR = Path("/mnt/data/gapfill_outputs_art_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
FREQ_MINUTES = 30

TARGET_COLS = [
    "co2_flux",
    "H",
    "LE",
    "ET",
]

PREDICTOR_MAP = {
    "co2_flux": ["H", "LE", "ET", "air_temperature", "RH", "VPD", "wind_speed", "u_star", "daytime", "DOY"],
    "H": ["LE", "co2_flux", "ET", "air_temperature", "RH", "VPD", "wind_speed", "u_star", "daytime", "DOY"],
    "LE": ["H", "co2_flux", "ET", "air_temperature", "RH", "VPD", "wind_speed", "u_star", "daytime", "DOY"],
    "ET": ["H", "LE", "co2_flux", "air_temperature", "RH", "VPD", "wind_speed", "u_star", "daytime", "DOY"],
}

# Reglas del esquema híbrido
MAX_INTERP_GAP_STEPS = 4
MAX_RF_GAP_STEPS = 16
STRUCTURAL_GAP_STEPS = 1700
MIN_SEGMENT_POINTS = 12

# Random Forest rápido pero suficientemente expresivo
RF_PARAMS = {
    "n_estimators": 30,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "random_state": 42,
    "n_jobs": -1,
}

VALIDATION_FRACTION = 0.15
N_VALIDATION_TRIALS = 10


## 2. Utilidades

In [ ]:
def find_runs(mask):
    runs = []
    start = None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            runs.append((start, i - 1))
            start = None
    if start is not None:
        runs.append((start, len(mask) - 1))
    return runs


def add_time_features(df, time_col):
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce", utc=True)
    out["hour"] = out[time_col].dt.hour
    out["minute"] = out[time_col].dt.minute
    out["dayofyear"] = out[time_col].dt.dayofyear

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24.0)
    out["minute_sin"] = np.sin(2 * np.pi * out["minute"] / 60.0)
    out["minute_cos"] = np.cos(2 * np.pi * out["minute"] / 60.0)
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 365.25)
    return out


def regularize_timeseries(df, time_col, freq_minutes):
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce", utc=True)
    out = out.dropna(subset=[time_col]).sort_values(time_col)

    if out[time_col].duplicated().any():
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
        other_cols = [c for c in out.columns if c not in numeric_cols and c != time_col]
        g_num = out.groupby(time_col, as_index=False)[numeric_cols].mean()
        if other_cols:
            g_other = out.groupby(time_col, as_index=False)[other_cols].first()
            out = g_num.merge(g_other, on=time_col, how="left")
        else:
            out = g_num

    full_index = pd.date_range(
        start=out[time_col].min(),
        end=out[time_col].max(),
        freq=f"{freq_minutes}min",
        tz="UTC"
    )

    out = out.set_index(time_col).reindex(full_index)
    out.index.name = time_col
    out = out.reset_index()
    return out


def classify_structure_combined(df, vars_for_structure, time_col, structural_gap_steps, min_segment_points):
    availability = df[vars_for_structure].notna().any(axis=1).to_numpy()
    gap_runs = find_runs(~availability)

    segment_id = np.full(len(df), 1.0)
    current = 1
    for start, end in gap_runs:
        gap_len = end - start + 1
        if gap_len > structural_gap_steps and end + 1 < len(df):
            current += 1
            segment_id[end + 1:] = current

    seg_series = pd.Series(segment_id, index=df.index, name="segment_id")

    summary_rows = []
    for seg in sorted(seg_series.dropna().unique()):
        seg = int(seg)
        idx = np.where(seg_series.values == seg)[0]
        n_points = len(idx)
        action = "main_segment" if n_points >= min_segment_points else "short_segment"

        summary_rows.append({
            "segment_id": seg,
            "start_idx": int(idx.min()),
            "end_idx": int(idx.max()),
            "n_points": int(n_points),
            "start_time": df.loc[idx.min(), time_col],
            "end_time": df.loc[idx.max(), time_col],
            "duration_hours": float((df.loc[idx.max(), time_col] - df.loc[idx.min(), time_col]).total_seconds() / 3600.0),
            "action_taken": action,
            "is_valid_for_analysis": True
        })

    return seg_series, pd.DataFrame(summary_rows)


def gap_fill_short_internal_gaps(series, max_gap_steps):
    s = series.copy()
    interp = s.interpolate(method="time", limit=max_gap_steps, limit_area="inside")

    flags = pd.Series("observed", index=s.index, dtype="object")
    mask = s.isna().to_numpy()

    for start, end in find_runs(mask):
        gap_len = end - start + 1
        left_bounded = start > 0 and pd.notna(s.iloc[start - 1])
        right_bounded = end < len(s) - 1 and pd.notna(s.iloc[end + 1])

        if left_bounded and right_bounded and gap_len <= max_gap_steps:
            flags.iloc[start:end + 1] = "imputed_time_limited"
        else:
            interp.iloc[start:end + 1] = np.nan
            if left_bounded and right_bounded:
                flags.iloc[start:end + 1] = "left_as_nan_large_gap"
            else:
                flags.iloc[start:end + 1] = "left_as_nan_edge_gap"

    return interp, flags


def _build_predictor_set(df, target_col, predictor_cols, time_col):
    out = add_time_features(df, time_col=time_col)
    base = [c for c in predictor_cols if c in out.columns and c != target_col]
    time_feats = ["hour", "minute", "dayofyear", "hour_sin", "hour_cos", "minute_sin", "minute_cos", "doy_sin", "doy_cos"]

    cols = []
    seen = set()
    for c in base + time_feats:
        if c not in seen and c in out.columns:
            cols.append(c)
            seen.add(c)
    return out, cols


def rf_gapfill_variable(df, target_col, predictor_cols, time_col, rf_params, validation_fraction=0.15):
    data, X_cols = _build_predictor_set(df, target_col, predictor_cols, time_col)

    observed_mask = data[target_col].notna()
    train_df = data.loc[observed_mask].dropna(subset=X_cols).copy()

    pred_mask = data[target_col].isna()
    pred_df = data.loc[pred_mask].dropna(subset=X_cols).copy()

    flags = pd.Series("observed", index=data.index, name=f"{target_col}_rf_flag", dtype="object")

    if len(train_df) < 30:
        metrics = pd.DataFrame([{
            "variable": target_col,
            "n_train": len(train_df),
            "n_predicted": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "status": "insufficient_training_data"
        }])
        importance = pd.DataFrame(columns=["variable", "feature", "importance"])
        out = df.copy()
        return out, flags, metrics, importance

    X_train = train_df[X_cols]
    y_train = train_df[target_col]

    rng = np.random.default_rng(rf_params.get("random_state", 42))
    train_idx = train_df.index.to_numpy()
    n_val = max(1, int(len(train_idx) * validation_fraction))
    val_idx = rng.choice(train_idx, size=n_val, replace=False)

    subtrain_df = train_df.drop(index=val_idx)
    valid_df = train_df.loc[val_idx]

    model = RandomForestRegressor(**rf_params)
    model.fit(subtrain_df[X_cols], subtrain_df[target_col])
    y_val_pred = model.predict(valid_df[X_cols])

    mae = mean_absolute_error(valid_df[target_col], y_val_pred)
    rmse = np.sqrt(mean_squared_error(valid_df[target_col], y_val_pred))
    r2 = r2_score(valid_df[target_col], y_val_pred) if len(valid_df) > 1 else np.nan

    model.fit(X_train, y_train)

    out = df.copy()
    if len(pred_df) > 0:
        y_pred = model.predict(pred_df[X_cols])
        out.loc[pred_df.index, target_col] = y_pred
        flags.loc[pred_df.index] = "imputed_rf"

    missing_predictors_idx = data.loc[pred_mask].index.difference(pred_df.index)
    if len(missing_predictors_idx) > 0:
        flags.loc[missing_predictors_idx] = "left_as_nan_missing_predictors"

    metrics = pd.DataFrame([{
        "variable": target_col,
        "n_train": int(len(train_df)),
        "n_predicted": int(len(pred_df)),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2) if pd.notna(r2) else np.nan,
        "status": "ok"
    }])

    importance = pd.DataFrame({
        "variable": target_col,
        "feature": X_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return out, flags, metrics, importance


def rf_gapfill_time_only(df, target_col, time_col, rf_params, validation_fraction=0.15):
    data = add_time_features(df, time_col=time_col)
    X_cols = ["hour", "minute", "dayofyear", "hour_sin", "hour_cos", "minute_sin", "minute_cos", "doy_sin", "doy_cos"]

    observed_mask = data[target_col].notna()
    train_df = data.loc[observed_mask].dropna(subset=X_cols).copy()

    pred_mask = data[target_col].isna()
    pred_df = data.loc[pred_mask].dropna(subset=X_cols).copy()

    flags = pd.Series("observed", index=data.index, name=f"{target_col}_rf_time_flag", dtype="object")

    if len(train_df) < 30:
        metrics = pd.DataFrame([{
            "variable": target_col,
            "n_train": len(train_df),
            "n_predicted": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "status": "insufficient_training_data"
        }])
        importance = pd.DataFrame(columns=["variable", "feature", "importance"])
        out = df.copy()
        return out, flags, metrics, importance

    X_train = train_df[X_cols]
    y_train = train_df[target_col]

    rng = np.random.default_rng(rf_params.get("random_state", 42) + 7)
    train_idx = train_df.index.to_numpy()
    n_val = max(1, int(len(train_idx) * validation_fraction))
    val_idx = rng.choice(train_idx, size=n_val, replace=False)

    subtrain_df = train_df.drop(index=val_idx)
    valid_df = train_df.loc[val_idx]

    model = RandomForestRegressor(**rf_params)
    model.fit(subtrain_df[X_cols], subtrain_df[target_col])
    y_val_pred = model.predict(valid_df[X_cols])

    mae = mean_absolute_error(valid_df[target_col], y_val_pred)
    rmse = np.sqrt(mean_squared_error(valid_df[target_col], y_val_pred))
    r2 = r2_score(valid_df[target_col], y_val_pred) if len(valid_df) > 1 else np.nan

    model.fit(X_train, y_train)

    out = df.copy()
    if len(pred_df) > 0:
        y_pred = model.predict(pred_df[X_cols])
        out.loc[pred_df.index, target_col] = y_pred
        flags.loc[pred_df.index] = "imputed_rf_time_only"

    metrics = pd.DataFrame([{
        "variable": target_col,
        "n_train": int(len(train_df)),
        "n_predicted": int(len(pred_df)),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2) if pd.notna(r2) else np.nan,
        "status": "ok"
    }])

    importance = pd.DataFrame({
        "variable": target_col,
        "feature": X_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return out, flags, metrics, importance


def hybrid_gapfill_segment(df_segment, target_col, predictor_cols, time_col, max_interp_gap_steps, max_rf_gap_steps, rf_params):
    out = df_segment.copy()

    temp = out.set_index(time_col)[target_col].copy()
    interp, interp_flags = gap_fill_short_internal_gaps(temp, max_gap_steps=max_interp_gap_steps)
    out[target_col] = interp.values
    flags = pd.Series(interp_flags.values, index=out.index, name=f"{target_col}_flag", dtype="object")

    remaining_mask = out[target_col].isna().to_numpy()
    remaining_runs = find_runs(remaining_mask)

    rf_candidate_idx = []
    for start, end in remaining_runs:
        gap_len = end - start + 1
        if gap_len <= max_rf_gap_steps:
            rf_candidate_idx.extend(range(start, end + 1))

    # RF con anclas
    if len(rf_candidate_idx) > 0:
        rf_out, rf_flags, rf_metrics, rf_importance = rf_gapfill_variable(
            df=out,
            target_col=target_col,
            predictor_cols=predictor_cols,
            time_col=time_col,
            rf_params=rf_params,
            validation_fraction=VALIDATION_FRACTION,
        )

        for idx in rf_candidate_idx:
            if pd.isna(out.iloc[idx][target_col]) and pd.notna(rf_out.iloc[idx][target_col]):
                out.iloc[idx, out.columns.get_loc(target_col)] = rf_out.iloc[idx][target_col]
                flags.iloc[idx] = "imputed_rf"

    # RF solo temporal como respaldo
    still_missing = out[target_col].isna()
    if still_missing.any():
        rf_time_out, rf_time_flags, rf_time_metrics, rf_time_importance = rf_gapfill_time_only(
            df=out,
            target_col=target_col,
            time_col=time_col,
            rf_params=rf_params,
            validation_fraction=VALIDATION_FRACTION,
        )

        newly_filled = still_missing & rf_time_out[target_col].notna()
        out.loc[newly_filled, target_col] = rf_time_out.loc[newly_filled, target_col]
        flags.loc[newly_filled] = "imputed_rf_time_only"

    still_missing = out[target_col].isna()
    flags.loc[still_missing & flags.eq("observed")] = "left_as_nan_large_gap"

    return out, flags


def validate_by_masking_hybrid(df_segment, target_col, predictor_cols, time_col, max_interp_gap_steps, max_rf_gap_steps, rf_params, n_trials=10, seed=42):
    s = df_segment[target_col].copy()
    values = s.to_numpy()
    rng = np.random.default_rng(seed)

    eligible = []
    for gap_len in range(1, max_rf_gap_steps + 1):
        for start in range(1, len(s) - gap_len - 1):
            block = values[start:start + gap_len]
            if np.all(pd.notna(block)) and pd.notna(values[start - 1]) and pd.notna(values[start + gap_len]):
                eligible.append((start, gap_len))

    if len(eligible) == 0:
        return {
            "variable": target_col,
            "n_trials": 0,
            "n_points_scored": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "r2": np.nan,
        }

    chosen = rng.choice(len(eligible), size=min(n_trials, len(eligible)), replace=False)

    truth_values = []
    pred_values = []

    for idx in chosen:
        start, gap_len = eligible[idx]
        sub = df_segment.copy()
        truth = sub.iloc[start:start + gap_len][target_col].copy()
        sub.iloc[start:start + gap_len, sub.columns.get_loc(target_col)] = np.nan

        pred_df, pred_flags = hybrid_gapfill_segment(
            df_segment=sub,
            target_col=target_col,
            predictor_cols=predictor_cols,
            time_col=time_col,
            max_interp_gap_steps=max_interp_gap_steps,
            max_rf_gap_steps=max_rf_gap_steps,
            rf_params=rf_params,
        )

        pred = pred_df.iloc[start:start + gap_len][target_col]
        valid = truth.notna() & pred.notna()

        if valid.any():
            truth_values.extend(truth[valid].tolist())
            pred_values.extend(pred[valid].tolist())

    truth_values = np.asarray(truth_values, dtype=float)
    pred_values = np.asarray(pred_values, dtype=float)

    if len(truth_values) == 0:
        return {
            "variable": target_col,
            "n_trials": int(len(chosen)),
            "n_points_scored": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "r2": np.nan,
        }

    mae = float(np.mean(np.abs(pred_values - truth_values)))
    rmse = float(np.sqrt(np.mean((pred_values - truth_values) ** 2)))
    bias = float(np.mean(pred_values - truth_values))
    denom = float(np.sum((truth_values - truth_values.mean()) ** 2))
    r2 = float(1 - np.sum((truth_values - pred_values) ** 2) / denom) if denom > 0 else np.nan

    return {
        "variable": target_col,
        "n_trials": int(len(chosen)),
        "n_points_scored": int(len(truth_values)),
        "mae": mae,
        "rmse": rmse,
        "bias": bias,
        "r2": r2,
    }


## 3. Carga y regularización del archivo de prueba

In [ ]:
df_raw = pd.read_csv(INPUT_CSV)
print("Dimensiones del archivo original:", df_raw.shape)
print("Primeras columnas:", df_raw.columns.tolist()[:15])
df_raw.head()


In [ ]:
df_regularized = regularize_timeseries(df_raw, time_col=TIME_COL, freq_minutes=FREQ_MINUTES)

candidate_numeric = set(TARGET_COLS)
for v in PREDICTOR_MAP.values():
    candidate_numeric.update(v)

for col in candidate_numeric:
    if col in df_regularized.columns:
        df_regularized[col] = pd.to_numeric(df_regularized[col], errors="coerce")

print("Dimensiones regularizadas:", df_regularized.shape)
df_regularized.head()


## 4. Segmentación estructural relajada

In [ ]:
segment_id, segment_summary = classify_structure_combined(
    df=df_regularized,
    vars_for_structure=[c for c in TARGET_COLS if c in df_regularized.columns],
    time_col=TIME_COL,
    structural_gap_steps=STRUCTURAL_GAP_STEPS,
    min_segment_points=MIN_SEGMENT_POINTS,
)

df_regularized["segment_id"] = segment_id
segment_summary


## 5. Gap filling híbrido por segmento y validación

In [ ]:
analysis_ready_dir = OUTPUT_DIR / "analysis_ready_segments"
analysis_ready_dir.mkdir(parents=True, exist_ok=True)

segment_outputs = []
flag_tables = []
validation_rows = []
method_counts = []

for seg in segment_summary["segment_id"].astype(int).tolist():
    sub_idx = df_regularized["segment_id"].eq(seg)
    sub = df_regularized.loc[sub_idx].copy().reset_index(drop=True)

    segment_flags = pd.DataFrame({TIME_COL: sub[TIME_COL]})

    for i, target_col in enumerate(TARGET_COLS):
        if target_col not in sub.columns:
            continue

        predictors = PREDICTOR_MAP.get(target_col, [])

        before_missing = int(sub[target_col].isna().sum())

        sub_filled, flags = hybrid_gapfill_segment(
            df_segment=sub,
            target_col=target_col,
            predictor_cols=predictors,
            time_col=TIME_COL,
            max_interp_gap_steps=MAX_INTERP_GAP_STEPS,
            max_rf_gap_steps=MAX_RF_GAP_STEPS,
            rf_params=RF_PARAMS,
        )

        after_missing = int(sub_filled[target_col].isna().sum())

        counts = flags.value_counts()
        method_counts.append({
            "segment_id": seg,
            "variable": target_col,
            "n_rows": len(sub),
            "missing_before": before_missing,
            "missing_after": after_missing,
            "filled_total": before_missing - after_missing,
            "filled_time_limited": int(counts.get("imputed_time_limited", 0)),
            "filled_rf": int(counts.get("imputed_rf", 0)),
            "filled_rf_time_only": int(counts.get("imputed_rf_time_only", 0)),
            "left_as_nan_large_gap": int(counts.get("left_as_nan_large_gap", 0)),
            "left_as_nan_edge_gap": int(counts.get("left_as_nan_edge_gap", 0)),
            "left_as_nan_missing_predictors": int(counts.get("left_as_nan_missing_predictors", 0)),
        })

        metrics = validate_by_masking_hybrid(
            df_segment=sub,
            target_col=target_col,
            predictor_cols=predictors,
            time_col=TIME_COL,
            max_interp_gap_steps=MAX_INTERP_GAP_STEPS,
            max_rf_gap_steps=MAX_RF_GAP_STEPS,
            rf_params=RF_PARAMS,
            n_trials=N_VALIDATION_TRIALS,
            seed=42 + i + seg,
        )
        metrics["segment_id"] = seg
        validation_rows.append(metrics)

        sub[target_col] = sub_filled[target_col].values
        segment_flags[f"{target_col}_flag"] = flags.values

    sub["segment_id"] = seg
    segment_outputs.append(sub)

    segment_flags["segment_id"] = seg
    flag_tables.append(segment_flags)

    ready_sub = sub.dropna(subset=[c for c in TARGET_COLS if c in sub.columns]).copy()
    ready_sub.to_csv(analysis_ready_dir / f"segment_{seg:02d}_analysis_ready.csv", index=False)

df_gapfilled_all = pd.concat(segment_outputs, ignore_index=True) if segment_outputs else pd.DataFrame()
df_gapfill_flags = pd.concat(flag_tables, ignore_index=True) if flag_tables else pd.DataFrame()
df_validation = pd.DataFrame(validation_rows) if validation_rows else pd.DataFrame()
df_method_counts = pd.DataFrame(method_counts) if method_counts else pd.DataFrame()

print("Segmentos detectados:", segment_summary["segment_id"].astype(int).tolist())
print("Archivos listos para análisis:", sorted([p.name for p in analysis_ready_dir.glob("*.csv")]))
df_method_counts


## 6. Exportación de resultados

In [ ]:
df_regularized.to_csv(OUTPUT_DIR / "art_regularized.csv", index=False)
segment_summary.to_csv(OUTPUT_DIR / "art_segment_summary.csv", index=False)
df_gapfilled_all.to_csv(OUTPUT_DIR / "art_gapfilled_all_segments.csv", index=False)
df_gapfill_flags.to_csv(OUTPUT_DIR / "art_gapfill_flags.csv", index=False)
df_validation.to_csv(OUTPUT_DIR / "art_gapfill_validation_summary.csv", index=False)
df_method_counts.to_csv(OUTPUT_DIR / "art_gapfill_method_counts.csv", index=False)

print("Resultados guardados en:", OUTPUT_DIR)


## 7. Gráficas de control

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
struct_var = TARGET_COLS[0] if TARGET_COLS else None

if struct_var is not None and struct_var in df_regularized.columns:
    ax.plot(df_regularized[TIME_COL], df_regularized[struct_var], ".", color="black", markersize=1.5, alpha=0.6, label="Serie regularizada")

for _, row in segment_summary.iterrows():
    color = "tab:green" if row["action_taken"] == "main_segment" else "tab:orange"
    ax.axvspan(row["start_time"], row["end_time"], color=color, alpha=0.10)

ax.set_title("Estructura temporal y segmentos detectados")
ax.set_xlabel("Tiempo")
ax.set_ylabel(struct_var if struct_var else "valor")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "structure_and_segments.png", dpi=300)
plt.show()


In [ ]:
for target_col in TARGET_COLS:
    if target_col not in df_gapfilled_all.columns:
        continue

    fig, ax = plt.subplots(figsize=(14, 4))

    raw = df_regularized[[TIME_COL, target_col]].copy()
    filled = df_gapfilled_all[[TIME_COL, target_col]].copy()

    ax.plot(raw[TIME_COL], raw[target_col], ".", color="black", markersize=1.5, alpha=0.6, label="Original/regularizado")
    ax.plot(filled[TIME_COL], filled[target_col], "-", color="tab:blue", linewidth=1.0, alpha=0.9, label="Gapfilled")

    flag_col = f"{target_col}_flag"
    if flag_col in df_gapfill_flags.columns:
        imputed_mask = df_gapfill_flags[flag_col].isin(["imputed_time_limited", "imputed_rf", "imputed_rf_time_only"])
        plot_df = df_gapfilled_all.loc[imputed_mask.values, [TIME_COL, target_col]].copy()
        ax.scatter(plot_df[TIME_COL], plot_df[target_col], s=8, color="tab:orange", alpha=0.8, label="Imputado")

    ax.set_title(f"Antes y después del gap filling: {target_col}")
    ax.set_xlabel("Tiempo")
    ax.set_ylabel(target_col)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{target_col}_before_after.png", dpi=300)
    plt.show()


In [ ]:
for target_col in TARGET_COLS:
    flag_col = f"{target_col}_flag"
    if flag_col not in df_gapfill_flags.columns or target_col not in df_gapfilled_all.columns:
        continue

    fig, ax = plt.subplots(figsize=(14, 2.8))

    original_missing = df_regularized[target_col].isna().astype(int)
    final_missing = df_gapfilled_all[target_col].isna().astype(int)

    ax.plot(df_regularized[TIME_COL], original_missing, label="Missing antes", linewidth=1.0)
    ax.plot(df_gapfilled_all[TIME_COL], final_missing, label="Missing después", linewidth=1.0)

    ax.set_title(f"Máscara de faltantes antes y después: {target_col}")
    ax.set_xlabel("Tiempo")
    ax.set_ylabel("Missing (0/1)")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{target_col}_missing_mask.png", dpi=300)
    plt.show()


In [ ]:
if not df_validation.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    df_validation.groupby("variable")["rmse"].mean().plot(kind="bar", ax=axes[0], title="RMSE promedio")
    axes[0].set_ylabel("RMSE")
    axes[0].grid(True, axis="y", alpha=0.3)

    df_validation.groupby("variable")["r2"].mean().plot(kind="bar", ax=axes[1], title="R² promedio")
    axes[1].set_ylabel("R²")
    axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "gapfill_validation_metrics.png", dpi=300)
    plt.show()


## 8. Resumen final

In [ ]:
summary = df_method_counts.merge(
    df_validation,
    on=["segment_id", "variable"],
    how="left"
).sort_values(["segment_id", "variable"]).reset_index(drop=True)

summary


## Interpretación sugerida

- `art_gapfilled_all_segments.csv` conserva toda la serie ya regularizada y procesada.
- `art_gapfill_flags.csv` permite rastrear qué puntos fueron observados, imputados por interpolación, imputados por RF o dejados como faltantes.
- `analysis_ready_segments/` contiene una o varias series listas para el notebook de análisis.
- `art_gapfill_validation_summary.csv` y `art_gapfill_method_counts.csv` documentan el desempeño de la imputación.

La recomendación es que el notebook de análisis posterior lea los archivos de `analysis_ready_segments/` y procese cada segmento por separado si en el futuro aparecieran más de uno.
